# IMDb review scraper

I used this notebook during my thesis to collect user reviews from IMDb. The scraper opens a movie page, loads additional reviews and extracts the review text, date, reviewer name and helpfulness votes.

This is a cleaned copy of the original thesis code. IMDb can change its page structure, so I am keeping it here to document the data-collection process rather than claiming that it will still run unchanged today.

## Setup

In [ ]:
import asyncio
import json
import re
from pathlib import Path

import nest_asyncio
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

# Needed when Playwright is run from a Jupyter notebook
nest_asyncio.apply()

## Movie metadata

In [ ]:
def movie_meta_from_jsonld(html: str):
    """Get the movie title and release date from the page metadata."""
    soup = BeautifulSoup(html, "lxml")
    title = ""
    release_date = ""

    for tag in soup.select('script[type="application/ld+json"]'):
        try:
            data = json.loads(tag.text.strip())
            items = data if isinstance(data, list) else [data]

            for item in items:
                if isinstance(item, dict) and item.get("@type") in {"Movie", "TVSeries", "CreativeWork"}:
                    title = item.get("name") or title
                    release_date = item.get("datePublished") or release_date
        except Exception:
            continue

    return title, release_date

## Scraper

In [ ]:
async def scrape_reviews_dom_async(
    title_id: str,
    clicks: int = 5,
    headless: bool = False
):

    # Handle the consent banner when it appears
    async def handle_consent(page):
        patterns = [
            r"Agree\s*&\s*proceed", r"Accept(\s*All)?(\s*Cookies)?",
            r"I\s*Agree", r"I\s*Accept", r"Continue", r"Got it", r"OK",
        ]
        for pattern in patterns:
            try:
                buttons = page.get_by_role("button", name=re.compile(pattern, re.I))
                for i in range(await buttons.count()):
                    button = buttons.nth(i)
                    if await button.is_visible():
                        await button.click()
                        await page.wait_for_timeout(200)
            except Exception:
                pass
        try:
            for frame in page.frames:
                url = frame.url or ""
                name = frame.name or ""
                if re.search(r"(consent|privacy|sp_message_iframe|sourcepoint)", url, re.I) or re.search(r"sp_message_iframe", name, re.I):
                    for pattern in patterns:
                        try:
                            buttons = frame.get_by_role("button", name=re.compile(pattern, re.I))
                            for i in range(await buttons.count()):
                                button = buttons.nth(i)
                                if await button.is_visible():
                                    await button.click()
                                    await page.wait_for_timeout(200)
                        except Exception:
                            continue
        except Exception:
            pass

    # Wait until the first reviews are loaded
    async def wait_for_any_review(page, timeout_ms=15000):
        selectors = [
            '[data-testid="review-card-parent"]',
            '[data-testid="review-container"]',
            '.review-container', '.lister-item',
            '.ipc-list .ipc-list-card', '[data-review-id]'
        ]
        loop = asyncio.get_running_loop()
        deadline = loop.time() + (timeout_ms / 1000)

        while loop.time() < deadline:
            for selector in selectors:
                try:
                    await page.wait_for_selector(selector, state="attached", timeout=800)
                    return True
                except Exception:
                    pass
            try:
                await page.mouse.wheel(0, 800)
            except Exception:
                pass
            await page.wait_for_timeout(250)
        return False

    # Expand the full review text where possible
    async def expand_readmore_and_spoilers(page):
        # Open shortened reviews
        for _ in range(4):
            expanded = False
            try:
                buttons = page.get_by_role("button", name=re.compile(r"(Show|Read)\s+more", re.I))
                for i in range(await buttons.count()):
                    button = buttons.nth(i)
                    try:
                        if await button.is_visible():
                            await button.scroll_into_view_if_needed()
                            await button.click()
                            expanded = True
                            await page.wait_for_timeout(200)
                    except Exception:
                        continue
            except Exception:
                pass
            if not expanded:
                break
        # Reveal reviews hidden behind spoiler buttons
        for _ in range(2):
            expanded = False
            try:
                spoiler_buttons = page.get_by_role("button", name=re.compile(r"spoiler", re.I))
                for i in range(await spoiler_buttons.count()):
                    button = spoiler_buttons.nth(i)
                    try:
                        if await button.is_visible():
                            await button.scroll_into_view_if_needed()
                            await button.click()
                            expanded = True
                            await page.wait_for_timeout(200)
                    except Exception:
                        continue
            except Exception:
                pass
            if not expanded:
                break

    # Count how many review cards are currently on the page
    async def get_review_count(page) -> int:
        try:
            return await page.evaluate("""() => {
                const sels = [
                  '[data-testid="review-card-parent"]',
                  '[data-testid="review-container"]',
                  '.review-container',
                  '.lister-item',
                  '.ipc-list .ipc-list-card',
                  '[data-review-id]'
                ];
                let n = 0;
                for (const s of sels) n = Math.max(n, document.querySelectorAll(s).length);
                return n;
            }""")
        except Exception:
            return 0

    # Close popups that can block the load-more button
    async def close_coachmarks_and_overlays(page):
        for pattern in [r"Close", r"Dismiss", r"Not\s*now", r"No\s*thanks", r"Maybe\s*later"]:
            try:
                buttons = page.get_by_role("button", name=re.compile(pattern, re.I))
                for i in range(await buttons.count()):
                    button = buttons.nth(i)
                    if await button.is_visible():
                        await button.click()
                        await page.wait_for_timeout(200)
            except Exception:
                pass
        try:
            await page.evaluate("""
                (() => {
                  const sels = [
                    '[data-testid="coachmark"]',
                    '.ipc-coachmark', '.ipc-coachmark__container',
                    '[class*="coachmark"]',
                    '[data-testid*="cookie"]', '[id*="cookie"]',
                    '[class*="cookie"]',
                    '[data-testid*="consent"]', '[id*="consent"]',
                    '[class*="consent"]'
                  ];
                  for (const s of sels) {
                    document.querySelectorAll(s).forEach(el => {
                      el.style.setProperty('display','none','important');
                      el.style.setProperty('visibility','hidden','important');
                      el.style.setProperty('pointer-events','none','important');
                    });
                  }
                })();
            """)
        except Exception:
            pass

    # Load the next batch of reviews
    async def load_next_batch(page, prev_count: int) -> tuple[bool, int]:
        # Scroll down first because IMDb loads content dynamically
        for _ in range(3):
            try:
                await page.mouse.wheel(0, 1800)
            except Exception:
                pass
            await page.wait_for_timeout(250)
        try:
            for _ in range(3):
                await page.keyboard.press("End")
                await page.wait_for_timeout(200)
        except Exception:
            pass

        # Clear anything covering the button
        await close_coachmarks_and_overlays(page)

        # Try a few versions of the load-more button
        candidates = [
            page.locator('[data-testid="load-more-trigger"]'),
            page.locator('#load-more-trigger'),
            page.get_by_role("button", name=re.compile(r"Load\\s+More|Show\\s+More|Load\\s+\\d+\\s+more", re.I)),
            page.get_by_text(re.compile(r"Load\\s+\\d+\\s+more|Load\\s+more|Show\\s+more", re.I)),
            page.locator('a.ipc-see-more, button.ipc-see-more'),
            page.locator('button:has-text("more")'),
        ]
        for loc in candidates:
            try:
                if await loc.count() > 0:
                    btn = loc.first
                    await btn.scroll_into_view_if_needed()
                    if await btn.is_visible():
                        await btn.click(force=True)
                        break
            except Exception:
                continue

        # Check whether new reviews actually appeared
        grew = False
        for _ in range(18):
            try:
                now = await get_review_count(page)
            except Exception:
                now = prev_count
            if now > prev_count:
                grew = True
                prev_count = now
                break
            await page.wait_for_timeout(350)

        # Expand any newly loaded reviews
        await expand_readmore_and_spoilers(page)
        await page.wait_for_timeout(200)
        return grew, prev_count

    # Open IMDb in Playwright
    start_url_title = f"https://www.imdb.com/title/{title_id}/"
    start_url_reviews = f"https://www.imdb.com/title/{title_id}/reviews?sort=submissionDate&dir=desc&ratingFilter=0"

    async with async_playwright() as p:
        user_agent = (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        )
        browser = await p.chromium.launch(headless=headless, args=["--disable-blink-features=AutomationControlled"])
        context = await browser.new_context(
            viewport={"width": 1366, "height": 1100},
            user_agent=user_agent,
            locale="en-US",
            timezone_id="Europe/Amsterdam",
            extra_http_headers={"Accept-Language": "en-US,en;q=0.8"}
        )
        context.set_default_timeout(30000)
        context.set_default_navigation_timeout(45000)
        page = await context.new_page()

        # First get the movie title and release date
        await page.goto(start_url_title, wait_until="domcontentloaded")
        await handle_consent(page)
        title_html = await page.content()
        movie_title, release_date = movie_meta_from_jsonld(title_html)

        # Then open the user-review page
        await page.goto(start_url_reviews, wait_until="domcontentloaded")
        await handle_consent(page)
        if "/consent" in page.url.lower() or "/privacy" in page.url.lower() or "/reviews" not in page.url:
            await page.goto(start_url_reviews, wait_until="domcontentloaded")
            await handle_consent(page)

        await wait_for_any_review(page, timeout_ms=15000)
        await expand_readmore_and_spoilers(page)

        # Load more reviews until the page stops growing
        prev = await get_review_count(page)
        for _ in range(clicks):
            grew, prev = await load_next_batch(page, prev)
            if not grew:
                break

        # Extract the fields I used later in the analysis
        rows = await page.evaluate("""() => {
            function t(el){ return el ? el.textContent.trim() : ""; }
            function pick(el, sels){ for(const s of sels){ const n = el.querySelector(s); if(n) return n; } return null; }
            function toIntK(s){
                if(!s) return 0;
                s = s.replace(/[,\\s]/g,'');
                const mk = s.match(/([\\d\\.]+)k/i);
                if(mk) return Math.round(parseFloat(mk[1]) * 1000);
                const n = s.match(/\\d+/g); return n ? parseInt(n.join(''),10) : 0;
            }
            const cards = Array.from(document.querySelectorAll(
                '[data-testid="review-card-parent"], [data-testid="review-container"], .review-container, .lister-item, .ipc-list .ipc-list-card, [data-review-id]'
            ));
            return cards.map(card => {
                const container = card.closest('article') || card;
                const body = pick(card, [
                    '[data-testid="review-overflow"] .ipc-html-content-inner-div',
                    'span[data-testid="review-text"]',
                    '[data-testid="review-primary-text"]',
                    '.text.show-more__control'
                ]);
                const title = pick(card, ['[data-testid="review-summary"] h3', '[data-testid="review-title"]', '.title', 'a.title']);
                const author = pick(container, [
                    'span[data-testid="review-author"]',
                    'a[href*="/user/"]',
                    '.display-name-link, .ipc-metadata-list-item__list-content-item'
                ]);
                const dateEl = pick(container, ['span[data-testid="review-date"]', '.review-date']);
                const up = card.querySelector('.ipc-voting__label__count--up');
                const down = card.querySelector('.ipc-voting__label__count--down');
                let helpful_votes = up ? toIntK(up.textContent) : 0;
                let not_helpful_votes = down ? toIntK(down.textContent) : 0;
                if(!up && !down){
                    const hb = card.querySelector('[data-testid="review-helpfulness"]');
                    if(hb){
                        const m = hb.innerText.trim().match(/(\\d[\\d\\.,Kk]*)\\s*out of\\s*(\\d[\\d\\.,Kk]*)/i);
                        if(m){
                            const toInt = (v)=>{ v=v.replace(/,/g,''); return /k$/i.test(v)? Math.round(parseFloat(v)*1000):parseInt(v,10); };
                            helpful_votes = toInt(m[1]); const total = toInt(m[2]); not_helpful_votes = Math.max(0, total - helpful_votes);
                        }else{
                            const m2 = hb.innerText.trim().match(/(\\d[\\d\\.,Kk]*)/);
                            if(m2){ helpful_votes = parseInt(m2[1].replace(/,/g,''),10) || 0; }
                        }
                    }
                }
                return {
                    review_text: t(body) || t(title),
                    reviewer_name: t(author),
                    review_date: t(dateEl),
                    helpful_votes, not_helpful_votes
                };
            });
        }""")

        await browser.close()
        return movie_title, release_date, rows

## Run the scraper

In [ ]:
# Choose the movie and the maximum number of extra review batches to load
TITLE_ID = "tt6791350"
CLICKS = 10

movie_title, release_date, rows = asyncio.get_event_loop().run_until_complete(
    scrape_reviews_dom_async(
        TITLE_ID,
        clicks=CLICKS,
        headless=False
    )
)

df = pd.DataFrame(
    [
        {
            "movie_title": movie_title,
            "release_date": release_date,
            **row
        }
        for row in rows
    ],
    columns=[
        "movie_title",
        "release_date",
        "review_text",
        "reviewer_name",
        "review_date",
        "helpful_votes",
        "not_helpful_votes"
    ]
)

print(f"Collected {len(df):,} reviews for {movie_title or TITLE_ID}.")
df.head()

## Save the result

In [ ]:
# Raw scraped reviews stay outside the public repository
OUTPUT_DIR = Path("../data/private")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_file = OUTPUT_DIR / f"{TITLE_ID}_reviews.csv"
df.to_csv(output_file, index=False, encoding="utf-8")

print(f"Saved to {output_file}")